# L100 · The Multi-Agent Supervisor (build in the UI, drive from code)

In `01_agent_bricks_types` you built the six single-purpose Agent Bricks types and *previewed* the seventh — the
**Supervisor Agent**. This notebook builds it for real. The Supervisor is the standout of Agent Bricks: it does not
answer directly, it **orchestrates** — given a cross-domain question it decides which subagents to consult, runs them
over governed Unity Catalog data, and **fuses one answer** with a routing trace you can inspect.

You will do this the way Agent Bricks is meant to be used:

1. **Build it with clicks** — assemble a Supervisor over your three Akzo Genie spaces in the no-code UI.
2. **Drive it from code** — call the deployed Supervisor with the **Supervisor API**, and verify each subagent
   directly from the Genie Conversation API.

```
   question ─▶ SUPERVISOR ─▶ {Finance? SCM? Commercial?} ─▶ run chosen subagents ─▶ fuse ─▶ one answer
                   │                  routing                    (governed, OBO)        + routing trace
             reads each subagent's *description* to decide
```

> Note: all data is synthetic. Product names, accounts, suppliers, and documents are invented for the workshop.

**Reference docs:** [Multi-Agent Supervisor](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor) ·
[Supervisor API](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/supervisor-api)

## Where this sits in the workshop

The Supervisor reuses what you already built — it is only as good as the subagents it routes to.

```
  L100  00 SQL AI functions     ── call AI from SQL
        01 Agent Bricks types   ── build Genie, KA, Extraction, Parsing, Classification
        04 Supervisor (HERE)    ── orchestrate those into one cross-domain agent   ← you are here
  L200  governed supervisor in code, agents that act, MCP, eval
  L300  the deployable supervisor app (apps/supervisor/)
```

This is the no-code + API view. L200 chapter 1 reproduces the same route → run → fuse loop in pure Python so you can
see every moving part; L300 ships it as an app. Here you let **Agent Bricks run the loop for you**.

## Prerequisites

- The shared data is loaded (`../data/load_to_uc.py`) so the `akzo_finance` / `akzo_scm` / `akzo_commercial`
  tables exist.
- The **three domain Genie spaces** exist. Create them from code with `../genie/create_genie_spaces.py` (it prints
  and writes the ids to `genie/space_ids.json`), or by hand in the UI — see `../genie/README.md`. You meet them in
  `01_agent_bricks_types` section 1.
- Access to a chat model serving endpoint, and permission to **create an agent** in the workspace.
- A recent `databricks-sdk` (for the Genie Conversation API used to verify subagents).

In [ ]:
# MAGIC %pip install --quiet "databricks-sdk>=0.96"

In [ ]:
dbutils.library.restartPython()

## Setup — parameters

Widgets keep the notebook portable. Paste your three Genie space ids (the last URL segment of
`/genie/rooms/<space_id>`, or read them from `genie/space_ids.json`). Leave `supervisor_endpoint` blank until you
have built and deployed the Supervisor in Part 2 — then paste its serving-endpoint name to run Part 4a.

In [ ]:
dbutils.widgets.text("catalog", "", "Unity Catalog (blank = current_catalog())")
dbutils.widgets.text("llm_endpoint", "databricks-claude-sonnet-4-5", "Supervisor model endpoint")
dbutils.widgets.text("finance_space_id", "", "Finance Genie space id")
dbutils.widgets.text("scm_space_id", "", "SCM Genie space id")
dbutils.widgets.text("commercial_space_id", "", "Commercial Genie space id")
dbutils.widgets.text("supervisor_endpoint", "", "Deployed Supervisor serving-endpoint name (Part 4a)")

CATALOG = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
SUP_ENDPOINT = dbutils.widgets.get("supervisor_endpoint").strip()
SPACE_IDS = {
    "FINANCE":    dbutils.widgets.get("finance_space_id").strip(),
    "SCM":        dbutils.widgets.get("scm_space_id").strip(),
    "COMMERCIAL": dbutils.widgets.get("commercial_space_id").strip(),
}

import json, requests
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

print("Catalog            :", CATALOG)
print("Supervisor model   :", LLM_ENDPOINT)
print("Genie space ids    :", {k: (v[:8] + "…" if v else "(blank)") for k, v in SPACE_IDS.items()})
print("Supervisor endpoint:", SUP_ENDPOINT or "(not deployed yet — set after Part 2)")

# Part 1 — What a Supervisor is (and is not)

A Supervisor is **not** a bigger prompt. It is a manager with a roster of specialists. Each specialist (a
**subagent**) carries a one-line **description** — *what it is good at*. On every question the Supervisor:

1. **Routes** — reads the question plus the descriptions, decides which subagents are needed (often more than one).
2. **Runs** — calls each chosen subagent. Genie subagents write + run governed SQL **under the caller's identity**.
3. **Fuses** — composes one answer grounded only in what the subagents returned, and shows the routing trace.

| Your subagent | What it is good at (its *description*) |
|---|---|
| **Akzo Finance** (Genie) | gross margin, price, FX, COGS / raw-material / freight / energy cost, budget variance |
| **Akzo SCM** (Genie) | OTIF, inventory/stockouts, lanes, lead times, service levels, backorders |
| **Akzo Commercial** (Genie) | account churn risk, NPS, complaints, pipeline, sales by account/segment |

The description is the single most important thing you write — routing quality *is* description quality. Keep one
Supervisor focused (Databricks supports up to **30** tools/subagents; stay well under that — we use exactly 3).

# Part 2 — BUILD IT IN THE UI (no code)

This is the part you do with clicks. Each step ends with a **checkpoint** telling you what a working result looks
like. Screenshots are placeholders — drop your own into `images/` under the named file and they render inline.

## 2.0  Confirm your three subagents exist

The Supervisor routes to your three Akzo Genie spaces, so they must exist first. The cell below lists the Genie
spaces on your workspace and flags whether the three widget ids resolve. If any are missing, create them with
`../genie/create_genie_spaces.py` (or in the UI per `../genie/README.md`) and paste the ids into the widgets above.

In [ ]:
spaces = {s.space_id: s.title for s in (w.genie.list_spaces().spaces or [])}
print(f"{len(spaces)} Genie space(s) visible to you\n")
for dom, sid in SPACE_IDS.items():
    if not sid:
        print(f"  {dom:11s} widget blank  -> paste the space id (see ../genie/README.md)")
    elif sid in spaces:
        print(f"  {dom:11s} OK -> {spaces[sid]}")
    else:
        print(f"  {dom:11s} id {sid[:8]}… NOT found -> run ../genie/create_genie_spaces.py")
# Checkpoint: three OK lines. Those three spaces become the Supervisor's subagents.

## 2.1  Create the Supervisor Agent

![Create new Agent — Supervisor](images/supervisor_create.png)
> *Image: `images/supervisor_create.png` — the Create new Agent picker with **Supervisor Agent** selected.*

1. In the workspace, choose **New → Agent** (or open **Agents** and click **Create Agent**).
2. Choose **Supervisor Agent**.
3. Give it a **Description** that summarises the whole agent, for example:
   *"AkzoNobel coatings copilot. Answers cross-domain questions about margin, supply/service, and customer risk by
   routing to the Finance, SCM, and Commercial Genie spaces."*

**Checkpoint:** you land on the Supervisor configuration page with an empty **Tools and sub-agents** panel on the
left and a chat pane on the right.

## 2.2  Add the three Genie spaces as subagents (and describe each)

![Adding subagents with descriptions](images/supervisor_subagents.png)
> *Image: `images/supervisor_subagents.png` — the Tools and sub-agents panel with three Genie spaces added.*

In the **Tools and sub-agents** panel, click to add a subagent, choose **Genie Space**, search for each Akzo space,
and add all three. For **each** one, write a **description** — this is what the router reads. Paste these:

| Subagent | Description to paste |
|---|---|
| **Akzo Finance** | *Gross margin %, realized price per unit, FX translation, COGS and its raw-material / freight / energy / overhead breakdown, budget variance. Use for any "why did margin, price, or cost change" question.* |
| **Akzo SCM** | *OTIF (on-time-in-full), inventory, stockouts, days of supply, transport lanes, lead times, service levels, backorders. Use for supply, service, delivery, or fulfilment questions.* |
| **Akzo Commercial** | *Customer accounts, churn risk/score, NPS, complaints, sales/revenue by account, pipeline. Use for customer-risk, retention, or account-impact questions.* |

> **Access note.** A Genie subagent needs access to the space **and** its underlying catalog objects. Because the
> Supervisor runs with the **caller's** Unity Catalog permissions (OBO), the Finance row filter from L200 chapter 1
> still applies — a controller and an EMEA planner asking the same question get answers backed by different rows.

**Checkpoint:** three subagent tiles, each with a description. Hover a tile and click the external-link icon to
check its permissions if a later test says "no access".

## 2.3  Test in the chat pane, then the Playground

![Routing trace in the test pane](images/supervisor_test.png)
> *Image: `images/supervisor_test.png` — a cross-domain answer with its routing trace expanded.*

In the right-hand chat pane, ask the **flagship cross-domain question**:

> *Paints EMEA gross margin dropped about 8% in Q2 2026 — is it price, volume, or a supply/service issue, and what
> should I do?*

Expand the trace. You should see the Supervisor route to **Akzo Finance** *and* **Akzo SCM** (often **Commercial**
too), run each, and fuse one answer. Click **Open in Playground** for a roomier view and to grab code samples.

**Checkpoint:** the trace shows ≥2 subagents consulted; the fused answer names the **~8.9pp margin bridge**
(price / FX / raw-material) **and** the **Rotterdam OTIF dip to ~89% in May**. If it only calls one subagent, sharpen
that subagent's description (Part 2.2) — routing follows the description, not the wording of the question.

## 2.4  Add examples, then deploy

1. On the **Examples** tab, click **+ Add** and paste a few golden questions (the flagship above, plus a
   single-domain one like *"Which Paints EMEA accounts are at churn risk in June 2026?"*). Examples nudge routing.
2. When satisfied, **deploy** the Supervisor — this creates a serving endpoint with the task `agent/v1/responses`.
3. Copy the **endpoint name** and paste it into the `supervisor_endpoint` widget at the top of this notebook.

**Checkpoint:** the agent page shows an **Endpoint** with state **Ready**. You now have a deployed Supervisor you can
call from anywhere — which is exactly what Part 4 does.

# Part 3 — Verify the subagents from code (Genie Conversation API)

Before driving the whole Supervisor, confirm each subagent answers correctly on its own. The **Genie Conversation
API** is exactly what the Supervisor calls under the hood for a Genie subagent: it sends a question to a space,
Genie writes + runs the governed SQL **under your identity**, and returns the SQL, the rows, and a narration.

This runs even before you deploy the Supervisor — it is your reference for what each leg should say.

In [ ]:
def genie_ask(space_id: str, question: str) -> dict:
    """Ask one Genie subagent. Returns {sql, rows, text} — Genie generates + runs governed SQL under your UC identity."""
    msg = w.genie.start_conversation_and_wait(space_id=space_id, content=question)
    sql, rows, text = None, [], []
    for att in (msg.attachments or []):
        if getattr(att, "text", None) and att.text.content:
            text.append(att.text.content)
        if getattr(att, "query", None) is not None:
            sql = att.query.query
            res = w.genie.get_message_attachment_query_result(
                space_id=space_id, conversation_id=msg.conversation_id,
                message_id=msg.id, attachment_id=att.attachment_id)
            sr = getattr(res, "statement_response", None)
            if sr and sr.result and sr.result.data_array:
                cols = [c.name for c in sr.manifest.schema.columns]
                rows = [dict(zip(cols, r)) for r in sr.result.data_array]
    return {"sql": sql, "rows": rows, "text": " ".join(text)}

**Verify the Finance subagent** — the number the whole story hangs on. Expect Genie to return the certified margin %
stepping down Q1 → Q2 2026.

In [ ]:
fin = SPACE_IDS["FINANCE"]
if fin:
    r = genie_ask(fin, "Show Decorative Paints EMEA gross margin percent by quarter for 2026.")
    print("Generated SQL:\n", (r["sql"] or "(none)")[:400], "\n")
    display(spark.createDataFrame(r["rows"]) if r["rows"] else spark.createDataFrame([{"note": r["text"][:300]}]))
    print("\nNarration:", r["text"][:300])
else:
    print("Set finance_space_id to verify this subagent.")
# Expect: 2026-Q1 ~39.6%, 2026-Q2 ~30.7% (the ~8.9pp drop the Supervisor must explain).

**Verify the SCM subagent** — the supply cause behind the margin shock. Expect the Rotterdam → EMEA-DACH lane OTIF
to dip in May 2026.

In [ ]:
scm = SPACE_IDS["SCM"]
if scm:
    r = genie_ask(scm, "Show monthly OTIF percent for the Rotterdam-NL to EMEA-DACH lane in 2026.")
    print("Generated SQL:\n", (r["sql"] or "(none)")[:400], "\n")
    display(spark.createDataFrame(r["rows"]) if r["rows"] else spark.createDataFrame([{"note": r["text"][:300]}]))
else:
    print("Set scm_space_id to verify this subagent.")
# Expect ~96% Jan-Mar -> ~89% May -> ~93% June: the disrupted EMEA lane.

**Verify the Commercial subagent** — the customer consequence. Expect three EMEA accounts with churn_score > 0.7.

In [ ]:
com = SPACE_IDS["COMMERCIAL"]
if com:
    r = genie_ask(com, "Which Paints EMEA accounts have churn score above 0.7 in June 2026? Include owner rep and NPS.")
    print("Generated SQL:\n", (r["sql"] or "(none)")[:400], "\n")
    display(spark.createDataFrame(r["rows"]) if r["rows"] else spark.createDataFrame([{"note": r["text"][:300]}]))
else:
    print("Set commercial_space_id to verify this subagent.")
# Expect ACC0001 / ACC0002 / ACC0003 — all churn_score > 0.7.

**What you just checked:** each subagent, on its own, returns governed SQL + rows you can verify against the `# Expect`
comments (the workshop's known values). The Supervisor's only job is
to decide *which* of these to call for a given question and fuse their results — which is Part 4.

# Part 4 — Drive the Supervisor from code (the Supervisor API)

Two ways to call a Supervisor from code. **4a** calls the one you deployed in Part 2 — the production path. **4b**
shows the `databricks-openai` *inline-tools* form, where you declare the subagents in the request and Databricks runs
the route → run → fuse loop without you deploying anything first.

## 4a — Call the Supervisor you deployed (production path)

A deployed Supervisor is a serving endpoint with the `agent/v1/responses` task. You call its `/invocations` route
with a `responses`-style payload: `{"input": [{"role": "user", "content": "…"}]}`. The response carries the
final message **and** the routing trace (`function_call` items naming each subagent that was consulted).

Because the call runs with the **caller's configured** credentials (your user in this notebook; a service principal if run as a job), every subagent respects your Unity Catalog permissions (OBO) — the
same governance you saw in Part 3, now end-to-end through the Supervisor.

In [ ]:
import re
_host = w.config.host.rstrip("/")
_token = w.config.authenticate()["Authorization"].split(" ", 1)[1]

def ask_supervisor(question: str, endpoint: str | None = None) -> dict:
    """Call a deployed Multi-Agent Supervisor. Returns the parsed responses payload.
    Reads the supervisor_endpoint widget at call time so you can deploy, paste the name, and
    re-run just this cell. The endpoint name is validated so it cannot inject a different URL path."""
    endpoint = (endpoint or dbutils.widgets.get("supervisor_endpoint").strip())
    if not re.fullmatch(r"[A-Za-z0-9_-]+", endpoint or ""):
        raise ValueError(f"Set the supervisor_endpoint widget to a valid serving-endpoint name (got {endpoint!r}).")
    resp = requests.post(
        f"{_host}/serving-endpoints/{endpoint}/invocations",
        headers={"Authorization": f"Bearer {_token}", "Content-Type": "application/json"},
        json={"input": [{"role": "user", "content": question}]},
        timeout=300,
    )
    resp.raise_for_status()
    return resp.json()

def show_supervisor(out: dict) -> None:
    """Print the routing trace (which subagents were consulted) then the fused answer."""
    routed, answer = [], []
    for item in out.get("output", []):
        if item.get("type") == "function_call":
            routed.append(item.get("name", ""))
        elif item.get("type") == "message":
            for c in item.get("content", []):
                if c.get("type") in ("output_text", "text") and c.get("text"):
                    answer.append(c["text"])
    print("ROUTING TRACE — subagents consulted:")
    for r in routed:
        print("  •", r)
    print("\n" + "=" * 80 + "\nFUSED ANSWER\n" + "=" * 80)
    print(answer[-1] if answer else "(no final message — inspect the raw payload)")

### The flagship cross-domain question

This is the question a single subagent cannot answer — it needs Finance (the margin bridge) **and** SCM (the
Rotterdam shock). Watch the routing trace fan out, then read the one fused answer.

In [ ]:
FLAGSHIP = ("Paints EMEA gross margin dropped about 8% in Q2 2026 — is it price, volume, "
            "or a supply/service issue, and what should I do?")

if SUP_ENDPOINT:
    out = ask_supervisor(FLAGSHIP)
    show_supervisor(out)
else:
    print("Deploy the Supervisor in Part 2, then paste its endpoint name into the supervisor_endpoint widget.")
# Expect routing -> Finance + SCM (often + Commercial), and a fused answer naming the ~8.9pp margin bridge
# (price/FX/raw-material) AND the Rotterdam OTIF dip to ~89% in May.

### A single-domain question routes to one subagent

Routing is selective, not "always call everyone". A pure customer-risk question should route to **Commercial** only.

In [ ]:
if SUP_ENDPOINT:
    out = ask_supervisor("Which Paints EMEA accounts are at churn risk in June 2026, and who owns them?")
    show_supervisor(out)
else:
    print("Set supervisor_endpoint to run this.")
# Expect routing -> COMMERCIAL only.

## 4b — Inline tools with `databricks-openai` (no deploy needed)

The Supervisor API can also run the loop from a **single request** where you declare the subagents as `tools`. This
is great for prototyping a Supervisor before you commit to deploying one. Databricks manages the multi-turn loop:
the model picks a tool, Databricks runs it, feeds the result back, and repeats until it has an answer.

```python
from databricks_openai import DatabricksOpenAI
client = DatabricksOpenAI(use_ai_gateway=True)

resp = client.responses.create(
    model=LLM_ENDPOINT,
    input=[{"type": "message", "role": "user", "content": FLAGSHIP}],
    tools=[
        {"type": "genie_space", "name": "Akzo Finance",
         "genie_space": {"space_id": SPACE_IDS['FINANCE']}},
        {"type": "genie_space", "name": "Akzo SCM",
         "genie_space": {"space_id": SPACE_IDS['SCM']}},
        {"type": "genie_space", "name": "Akzo Commercial",
         "genie_space": {"space_id": SPACE_IDS['COMMERCIAL']}},
    ],
)
print(resp.output_text)
```

Other subagent/tool types use the same shape — `{"type": "uc_function", "uc_function": {"name": "cat.sch.fn"}}`,
`{"type": "knowledge_assistant", "knowledge_assistant": {"knowledge_assistant_id": "…"}}`,
`{"type": "serving_endpoint", "serving_endpoint": {"name": "…"}}`, `{"type": "web_search"}`.

> **Enablement note.** Inline `genie_space` / `knowledge_assistant` / `uc_function` tool types are a preview that your
> account team enables per workspace. If your workspace does not have it yet, the call below prints the exact error
> instead of failing the notebook — and Part 4a (the deployed endpoint) is the path that always works.

> **Routing caveat.** The inline `tools` form takes a `name` + `space_id` per subagent, not the rich
> per-subagent **description** you wrote in the UI (Part 2.2). So inline routing leans on the space names and the
> model's own judgement — it can route differently than your deployed Supervisor. For description-driven routing,
> the deployed endpoint (4a) is the faithful path.

In [ ]:
def try_inline_supervisor(question: str):
    """Best-effort inline-tools Supervisor call. Degrades gracefully if the preview is not enabled."""
    if not all(SPACE_IDS.values()):
        print("Set all three space-id widgets to try the inline path."); return
    try:
        from databricks_openai import DatabricksOpenAI
    except ImportError:
        print("pip install databricks-openai to try the inline-tools path (Part 4a needs nothing extra)."); return
    client = DatabricksOpenAI(use_ai_gateway=True)
    tools = [{"type": "genie_space", "name": f"Akzo {d.title()}", "genie_space": {"space_id": sid}}
             for d, sid in SPACE_IDS.items()]
    try:
        resp = client.responses.create(
            model=LLM_ENDPOINT,
            input=[{"type": "message", "role": "user", "content": question}],
            tools=tools,
        )
        print(resp.output_text)
    except Exception as e:
        print("Inline-tools path not available on this workspace:\n ", str(e)[:300])
        print("\nUse Part 4a (the deployed Supervisor endpoint) — that path needs no preview.")

try_inline_supervisor(FLAGSHIP)

# What you built

- A **Multi-Agent Supervisor** over your three Akzo Genie spaces, assembled with **clicks** — no orchestration code.
- Verified each **subagent** from the Genie Conversation API (your reference for the expected numbers).
- Drove the deployed Supervisor with the **Supervisor API** and watched it **route → run → fuse** a cross-domain
  answer, governed per user via OBO.

| Path | When to use it |
|---|---|
| **Part 2 UI build** | the default — author, test, and deploy a Supervisor with no code |
| **Part 4a deployed endpoint** | production — call your deployed Supervisor from any app or notebook |
| **Part 4b inline tools** | prototyping — declare subagents in one request, no deploy (preview-gated) |

### Next
You built the Supervisor as a managed product here. In `../L200-capabilities/01_governed_supervisor.py` you build
the **same** route → run → fuse loop in pure Python — the router, the fuser, the OBO row filter — so nothing about
the Supervisor is a black box. Then L300 ships it as the deployable app in `../apps/supervisor/`.